In [ ]:
"""
NER Dataset Generator
=====================
Generates labelled NER training examples from template arrays.

Rules
-----
- Templates are consumed **in sequence**, cycling back to index 0 when exhausted.
- Each entity value array is also consumed **in sequence** (cycling).
- After a text is built, character-level start/end positions are computed and
  verified; any example where a value cannot be located unambiguously is
  added to a `flagged` list instead of the clean output.
- `generate(templates, entity_values, intent, count)` is the main entry-point.
"""

import json
import re


# ──────────────────────────────────────────────────────────────────────────────
# Internal helpers
# ──────────────────────────────────────────────────────────────────────────────

class _Cycler:
    """Infinite sequential cycling over a list (no randomness)."""

    def __init__(self, items: list):
        if not items:
            raise ValueError("Entity value list must not be empty.")
        self._items = items
        self._idx = 0

    def next(self):
        value = self._items[self._idx]
        self._idx = (self._idx + 1) % len(self._items)
        return value


def _find_span(text: str, value: str):
    """
    Return (start, end) of *value* inside *text* (case-insensitive search).
    Returns None when the value is not found or appears more than once
    (ambiguous → flag).
    """
    lower_text = text.lower()
    lower_value = value.lower()
    matches = [m.start() for m in re.finditer(re.escape(lower_value), lower_text)]
    if len(matches) != 1:
        return None
    start = matches[0]
    return start, start + len(value)


def _build_example(template: str, entity_labels: list, cyclers: dict, intent: str):
    """
    Fill one template, resolve positions, and return a dataset example dict.
    Returns None if any entity value cannot be located unambiguously.
    """
    # ── 1. Pick values for each slot in the template ──────────────────────────
    slot_values = {}          # label → chosen value (one per label per example)
    for label in entity_labels:
        if label not in cyclers:
            raise KeyError(
                f"No value array supplied for entity '{label}'. "
                f"Add it to the entity_values dict."
            )
        if label not in slot_values:          # only advance cycler once per label per example
            slot_values[label] = cyclers[label].next()

    # ── 2. Fill the template string ───────────────────────────────────────────
    text = template
    for label, value in slot_values.items():
        text = text.replace("{" + label + "}", value, 1)

    # ── 3. Resolve character positions ───────────────────────────────────────
    entities = []
    seen_labels = []          # track order of first occurrence in template

    for label in entity_labels:
        if label in seen_labels:
            continue          # skip duplicates (shouldn't happen, but be safe)
        seen_labels.append(label)

        value = slot_values[label]
        span = _find_span(text, value)
        if span is None:
            return None       # ambiguous / missing → caller will flag

        entities.append({
            "value": value,
            "entity": label,
            "start": span[0],
            "end": span[1],
        })

    # Sort entities by their start position (natural reading order)
    entities.sort(key=lambda e: e["start"])

    return {
        "text": text,
        "intent": intent,
        "entities": entities,
    }


# ──────────────────────────────────────────────────────────────────────────────
# Public API
# ──────────────────────────────────────────────────────────────────────────────

def generate(templates: list, entity_values: dict, intent: str, count: int):
    """
    Generate `count` NER training examples.

    Parameters
    ----------
    templates : list of (template_str, [entity_labels])
        Your ADD_TEMPLATES / DELETE_TEMPLATES / … arrays.
    entity_values : dict  {label: [value, value, …]}
        One list per entity type.  Every list cycles sequentially.
    intent : str
        Intent label written into every example  (e.g. "add", "delete").
    count : int
        Number of examples to generate.

    Returns
    -------
    clean   : list – valid examples ready for training
    flagged : list – examples where a position could not be resolved
    """
    if not templates:
        raise ValueError("`templates` must not be empty.")
    if count <= 0:
        raise ValueError("`count` must be a positive integer.")

    # One cycler per entity label
    cyclers = {label: _Cycler(values) for label, values in entity_values.items()}

    template_cycler = _Cycler(templates)   # sequential cycling over templates

    clean   = []
    flagged = []

    for _ in range(count):
        template_str, entity_labels = template_cycler.next()
        example = _build_example(template_str, entity_labels, cyclers, intent)

        if example is None:
            # Re-build without positions so we can inspect what went wrong
            slot_values = {lbl: cyclers[lbl]._items[(cyclers[lbl]._idx - 1) % len(cyclers[lbl]._items)]
                           for lbl in entity_labels}
            text = template_str
            for lbl, val in slot_values.items():
                text = text.replace("{" + lbl + "}", val, 1)
            flagged.append({
                "text": text,
                "intent": intent,
                "entities": [{"value": slot_values[lbl], "entity": lbl} for lbl in entity_labels],
                "_flag_reason": "value not found or ambiguous in text",
            })
        else:
            clean.append(example)

    return clean, flagged


def save_dataset(clean: list, flagged: list, clean_path: str, flagged_path: str = None,
                 append: bool = True):
    """
    Persist the generated dataset to JSON files.

    Parameters
    ----------
    clean        : clean examples from generate()
    flagged      : flagged examples from generate()
    clean_path   : output path for the clean dataset
    flagged_path : output path for flagged examples  (None → skip)
    append       : if True and clean_path exists, merge with existing data
    """
    import os

    existing = []
    if append and os.path.exists(clean_path):
        with open(clean_path, "r", encoding="utf-8") as f:
            existing = json.load(f)
        print(f"Loaded {len(existing)} existing examples from '{clean_path}'")

    combined = existing + clean
    with open(clean_path, "w", encoding="utf-8") as f:
        json.dump(combined, f, indent=2, ensure_ascii=False)
    print(f"Saved {len(combined)} clean examples → '{clean_path}'")

    if flagged_path and flagged:
        with open(flagged_path, "w", encoding="utf-8") as f:
            json.dump(flagged, f, indent=2, ensure_ascii=False)
        print(f"Saved {len(flagged)} flagged examples → '{flagged_path}'")
    elif flagged:
        print(f"Warning: {len(flagged)} flagged examples were NOT saved (no flagged_path given).")


# ──────────────────────────────────────────────────────────────────────────────
# Example usage  (replace arrays below with your real ones)
# ──────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    # ── paste your real arrays here ──────────────────────────────────────────
    # from NER_Templates_Python import ADD_TEMPLATES,SHOW_TEMPLATES,DELETE_TEMPLATES,UPDATE_TEMPLATES,BEHAVIORAL_TEMPLATES   # your template file
    

    collaborators = ['Aamir', 'Abbas', 'Abdul', 'Abdul Aziz', 'Abdul Haq', 'Abdullah', 'Abeeha', 'Abhishek', 'Abid', 'Abida', 'Abiha', 'Abrar', 'Adil', 'Adnan', 'Afzal', 'Ahmad', 'Ahmed', 'Ahsan', 'Aisha', 'Aiza', 'Aizah', 'Ajeet', 'Ajit', 'Akbar', 'Akhtar', 'Akif', 'Akmal', 'Akram', 'Alam', 'Ali', 'Aliya', 'Almas', 'Amanullah', 'Amar', 'Amar', 'Ameer', 'Amin', 'Amina', 'Amir', 'Amit', 'Amjad', 'Ammar', 'Amna', 'Amrita', 'Anas', 'Anil', 'Ansar', 'Anwar', 'Aqeel', 'Aqil', 'Aqsa', 'Areej', 'Arif', 'Arshad', 'Arun', 'Asad', 'Asadullah', 'Asees', 'Asghar', 'Ashfaq', 'Ashraf', 'Asif', 'Asim', 'Asiya', 'Asma', 'Atif', 'Aurangzeb', 'Ayaz', 'Ayesha', 'Aysha', 'Ayub', 'Azhar', 'Aziz', 'Azra', 'Babar', 'Baber', 'Babur', 'Bakht', 'Bakhtiar', 'Basheer', 'Bashir', 'Benazir', 'Bilal', 'Bismillah', 'Bushra', 'Danish', 'Daniyal', 'Daud', 'Deep', 'Deepa', 'Deepak', 'Dilshad', 'Dinesh', 'Dip', 'Dipa', 'Dipak', 'Ehsan', 'Esmatullah', 'Faheem', 'Fahmida', 'Faisal', 'Faiz', 'Faiza', 'Farah', 'Fareed', 'Fareeha', 'Farhan', 'Farhana', 'Farid', 'Farida', 'Fariha', 'Farooq', 'Farrukh', 'Farzana', 'Fateh', 'Fatima', 'Fawzia', 'Faysal', 'Fazal', 'Feroz', 'Feroze', 'Fida', 'Firdaus', 'Firdos', 'Firdous', 'Fouzia', 'Furqan', 'Ghulam', 'Gohar', 'Gul', 'Gulbadan', 'Gulbahar', 'Gulnaz', 'Gulrukh', 'Gulshan', 'Gulzar', 'Habib', 'Habibullah', 'Hafeez', 'Hafiz', 'Hafsa', 'Haider', 'Halima', 'Hameed', 'Hamid', 'Hamidullah', 'Haneef', 'Hanif', 'Haris', 'Haroon', 'Hasan', 'Haseeb', 'Hashim', 'Hasib', 'Hassan', 'Hayat', 'Hira', 'Humaira', 'Hussain', 'Hyder', 'Ibrahim', 'Ihsan', 'Ikram', 'Ilyas', 'Iman', 'Imran', 'Imtiaz', 'Imtiyaz', 'Inaaya', 'Inam', 'Inaya', 'Inayat', 'Inayatullah', 'Iqbal', 'Iqra', 'Irfan', 'Irshad', 'Ishfaq', 'Ismail', 'Ismat', 'Ismatullah', 'Jahangir', 'Jalal', 'Jamal', 'Jamil', 'Jamila', 'Jamshed', 'Jannat', 'Javaid', 'Javed', 'Juma', 'Junaid', 'Kamal', 'Kamal', 'Kamran', 'Karan', 'Kausar', 'Khadija', 'Khaled', 'Khalid', 'Khalida', 'Khalil', 'Khaliq', 'Khan', 'Khawaja', 'Khurram', 'Khurshid', 'Khwaja', 'Kiran', 'Kulsoom', 'Kulsum', 'Kumar', 'Kunal', 'Laila', 'Latif', 'Lubna', 'Luqman', 'Madan', 'Mahmood', 'Mahmud', 'Majid', 'Malala', 'Malalai', 'Maliha', 'Malik', 'Mangal', 'Manish', 'Manohar', 'Manoj', 'Mansoor', 'Manzoor', 'Maqsood', 'Maqsud', 'Mariyam', 'Maryam', 'Masood', 'Masooma', 'Masuma', 'Mehmood', 'Mehmud', 'Midhat', 'Mina', 'Mirza', 'Mohammad', 'Mohit', 'Mohsin', 'Muhammad', 'Muhammad Ali', 'Mukhtar', 'Mumtaz', 'Munawar', 'Munawara', 'Muneer', 'Munir', 'Murad', 'Murtaza', 'Musa', 'Muslim', 'Mustafa', 'Muzaffar', 'Nabeel', 'Nadeem', 'Nadia', 'Nadim', 'Nadir', 'Naeem', 'Naeema', 'Naila', 'Najeeb', 'Najiba', 'Najibullah', 'Najma', 'Nand', 'Naqibullah', 'Naresh', 'Nargis', 'Naseem', 'Naseer', 'Nasim', 'Nasir', 'Nasreen', 'Naveed', 'Navneet', 'Nazar', 'Nazeer', 'Nazia', 'Nazir', 'Nazir', 'Neha', 'Nida', 'Nimatullah', 'Nitish', 'Noor', 'Nur', 'Nusrat', 'Pankaj', 'Parvaiz', 'Parveen', 'Parvez', 'Parvin', 'Pervez', 'Pooja', 'Pritam', 'Puja', 'Qadir', 'Qamar', 'Qasim', 'Qurban', 'Rabia', 'Rafiq', 'Rahat', 'Rahim', 'Rahman', 'Rahmat', 'Rahmatullah', 'Rahul', 'Raj', 'Raja', 'Rajesh', 'Rakesh', 'Raman', 'Ramzan', 'Rana', 'Rani', 'Rasheed', 'Rashid', 'Rashida', 'Rasool', 'Raza', 'Razia', 'Rehana', 'Rehman', 'Rehmat', 'Ritu', 'Rizwan', 'Rizwana', 'Rubab', 'Rukhsana', 'Saadia', 'Saba', 'Sabeeha', 'Sabeen', 'Sabiha', 'Sabir', 'Sadaf', 'Saddam', 'Sadia', 'Sadiq', 'Saeed', 'Saeeda', 'Sagar', 'Sahib', 'Sahil', 'Saif', 'Saifullah', 'Saima', 'Saira', 'Sajida', 'Sajjad', 'Sakina', 'Salamat', 'Saleem', 'Salim', 'Salma', 'Salman', 'Samar', 'Sami', 'Samia', 'Samina', 'Sana', 'Sandeep', 'Sandip', 'Sanjeev', 'Sanjiv', 'Santosh', 'Sardar', 'Sayfullah', 'Shabana', 'Shabnam', 'Shafiq', 'Shaheen', 'Shahid', 'Shahida', 'Shahjahan', 'Shahnaz', 'Shahrukh', 'Shahzad', 'Shahzada', 'Shakir', 'Shama', 'Shameem', 'Shamim', 'Shams', 'Shandar', 'Sharif', 'Shaukat', 'Shazia', 'Shehnaz', 'Shekhar', 'Sher', 'Shiv', 'Shoaib', 'Shukria', 'Siddiq', 'Siddiqa', 'Sikandar', 'Simran', 'Sitara', 'Sohail', 'Subhan', 'Suhail', 'Sultan', 'Sultana', 'Sumeet', 'Sumit', 'Sunil', 'Swaran', 'Syed', 'Syeda', 'Tabassum', 'Tahir', 'Tahira', 'Taj', 'Talat', 'Talha', 'Talib', 'Tariq', 'Tarun', 'Tasleem', 'Taslim', 'Tasneem', 'Umair', 'Umar', 'Usman', 'Uzair', 'Varun', 'Vijay', 'Vinod', 'Vishal', 'Waheed', 'Waheeda', 'Wahid', 'Wahida', 'Wali', 'Waqar', 'Waqas', 'Waqqas', 'Waseem', 'Yama', 'Yaqoob', 'Yaseen', 'Yash', 'Yasin', 'Yasir', 'Yasmeen', 'Yasmin', 'Yogesh', 'Younus', 'Yousaf', 'Yousuf', 'Yusuf', 'Zafar', 'Zaheer', 'Zahid', 'Zahida', 'Zahir', 'Zahra', 'Zain', 'Zainab', 'Zakia', 'Zaman', 'Zamir', 'Zareen', 'Zarina', 'Zarmina', 'Zawar', 'Zeenat', 'Zehra', 'Zia', 'Ziauddin', 'Zohra', 'Zorawar', 'Zubaida', 'Zubair', 'Zuhra', 'Zulekha', 'Zulfaqar', 'Zulfikar', 'Zulfiqar']
    task_names= ['Social media campaign project', 'Practice linear regression', 'Revise Global Warming Data', 'Review Data Structures Notes', 'Study international trade', 'Mentorship session', 'Macroeconomics test', 'Physics II Final Exam', 'Biochemistry quiz', 'Study Group for Finals', 'History Documentary Script', 'Review Financial Statements', 'Submit workshop RSVP', 'Library Research Consultation', 'Supply chain simulation', 'Applied statistics midterm', 'Volunteering Orientation', 'Online Quiz 2 Closing', 'Network security report', 'Study Geotechnical formulas', 'Final Project Video Submission', 'Nursing Clinical Simulation', 'Practice Vector Calculus', 'Submit lab safety form', 'Part-time Job Briefing', 'Physics midterm', 'Memorize periodic table', 'Epidemiology final', 'Pay tuition fees', 'Post on discussion forum', 'Digital Marketing workshop', 'Fluid Mechanics Final', 'Upload coding repository', 'Revise networks chapter', 'Sports team practice', 'Applied Math tutorial', 'Financial Markets Lecture', 'Social Media Marketing Case Study', 'Embedded Systems lab', 'Financial Accounting Ledger Entry', 'Cell Biology exam', 'Read philosophy primary source', 'Study Control Systems', 'Artificial Intelligence Seminar', 'Environmental science lab', 'Number theory quiz', 'Constitutional Law Exam', 'Solve Integration problems', 'Embedded C coding task', 'Read Philosophy of Mind', 'Academic advisor meeting', 'Capston project presentation', 'Fashion Design portfolio', 'Read business ethics text', 'Genetics problem set', 'Game Dev prototype', 'Submit transcript request', 'Confirm graduation audit', 'Embedded systems project', 'Linear Algebra Final', 'Submit music composition', 'Philosophy midterm', 'Submit industrial training report', 'Graduation Application Form', 'Submit peer evaluation', 'Review Nursing Ethics', 'Forensic Science Lab', 'Internship Report Submission', 'Register for next semester', 'Re-watch finance lecture', 'Botany final exam', 'Human Resource Management Lecture', 'Machine Learning Model Training', 'Psychology Research Proposal', 'Ethics Committee Review', 'Practice Integration Techniques', 'Departmental seminar', 'Embedded Systems PCB Layout', 'Review labor economics', "Dean's list ceremony", 'Review tectonic plate theory', 'Complete diversity training', 'Sociology field notes', 'Non-profit strategy', 'Organic synthesis report', 'Cybersecurity audit project', 'Student council session', 'Public health report', 'Mechanical Drawing Exercise', 'Neuroscience Lab', 'Submit ethics approval form', 'Java programming task', 'Toxicology exam', 'Virology midterm', 'Marketing Plan Presentation', 'Criminal Profiling Group Project', 'Practice vector calculus', 'Immunology test', 'Zoology lab session', 'Graduate School Prep Meeting', 'Business Law case study', 'Study animal behavior', 'TCP/IP configuration lab', 'Financial aid consultation', 'Cell Biology lab report', 'Hydrology Mapping Project', 'Submit volunteer hours', 'Digital logic design lab', 'Mobile app unit testing', 'Linear Algebra quiz', 'Astronomy observatory session', 'Public Relations Campaign', 'Upload portfolio website', 'Mechatronics System Integration', 'Submit ethics clearance form', 'Grant proposal sync', 'Signal and Systems Tutorial', 'Read Criminal Law statutes', 'Dormitory floor meeting', 'Astronomy lecture', 'Finalize team contract', 'Organic Chemistry quiz', 'Read International Trade Theory', 'Corporate Finance lecture', 'Database normalization task', 'Study group meetup', 'Molecular Biology lab', 'Linguistics analysis', 'Data Science tutorial', 'Read poetry anthology', 'Library research group', 'Read jazz history notes', 'Modern History final', 'Digital marketing workshop', 'Fashion design portfolio', 'Political Science test', 'Regression analysis report', 'Study for algorithms', 'Media Ethics Response Paper', 'Study Metabolic Pathways', 'Study Circuit Analysis Laws', 'Peer Feedback Submission', 'Social Work Community Project', 'Nursing Care Plan Assignment', 'Study Fluid Power Systems', 'Biochemistry exam', 'Upload video presentation', 'Moot court session', 'Civil Engineering Bridge Design', 'Sign off on lab safety', 'Practice differential equations', 'Human-Computer Interaction lab', 'Global health workshop', 'E-commerce website build', 'Electromagnetism Final', 'Workshop on LaTeX', 'Supervisor meeting', 'Contemporary Art Seminar', 'TA office hours', 'Bridge design project', 'Professor Office Hours', 'Strategic management class', 'Review anatomy flashcards', 'Practice Spanish verbs', 'Course Evaluation Deadline', 'Upload presentation slides', 'Historical documentary project', 'Quantum Mechanics Midterm', 'Case study group discussion', 'Architecture Studio Model', 'Critical Thinking Essay', 'Practice Architectural Drafting', 'Advanced Calculus Seminar', 'Software engineering lecture', 'Cloud computing deployment', 'Research assistant meeting', 'Human Rights law brief', 'World History 101 Midterm', 'C++ programming assignment', 'Assembly language lab', 'Thermodynamics Midterm', 'Introduction to Sociology Lecture', 'International Relations simulation', 'Entrepreneurship Workshop', 'Summarize sociology readings', 'E-commerce Website Frontend', 'Post weekly blog entry', 'Digital Logic Design lab', 'Read supply chain case', 'Revise Ecology lecture', 'Case Study Group Sync', 'Horticulture lecture', 'Read political theory', 'Meeting with Thesis Advisor', 'Study Ancient Civilizations', 'Grant Application Deadline', 'Corporate Finance Midterm', 'App Development Team Standup', 'Supply Chain Management Analysis', 'Forensic pathology lecture', 'Endocrinology final', 'Renewable Energy Feasibility Study', 'Constitutional Law midterm', 'Submit thesis', 'Drop/Add Deadline', 'Submit scholarship application', 'Mechanical CAD project', 'Course Registration Consultation', 'Abstract algebra exam', 'UX/UI design wireframes', 'International Law Lecture', 'Calculus I Midterm', 'Compiler construction project', 'Genomics data project', 'Differential Equations midterm', 'Student Club Executive Meeting', 'Review Linguistics theories', 'Agile sprint task', 'Turn in field notes', 'Study Behavioral Economics', 'Finalize degree plan', 'Academic Advisor Session', 'Criminal law seminar', 'Biochemistry Final Exam', 'Study Human Bone Structure', 'Introduction to Psychology Midterm', 'Apply for exchange program', 'Resume building session', 'Fashion Design Studio', 'International Relations Final', 'Creative writing tutorial', 'Group project meeting', 'English Literature critique', 'Medieval history lecture', 'Political Science Reflection Paper', 'Quantum physics worksheet', 'Urban sociology lecture', 'Mental health counselor meeting', 'Psychology 101 quiz', 'Submit internship application', 'Robotics team assembly', 'Pay semester tuition', 'Political Science paper', 'Comparative Literature Lecture', 'Statics problem set', 'Circuit analysis problem set', 'Review organic reactions', 'Teaching assistant office hours', 'Faculty Research Discussion', 'Anthropology seminar', 'Psychology 101 midterm', 'Physiology midterm', 'Numerical analysis quiz', 'Practice Java Syntax', 'Aeronautical Wing Analysis', 'Sociology of Religion lecture', 'Review Business Statistics Formulas', 'Classical Mechanics Lecture', 'Study econometrics', 'Linear Algebra final', 'Civil Engineering model design', 'Bioinformatics Pipeline Project', 'Review Cryptography Algorithms', 'Internship coordinator meeting', 'Bioinformatics sequence project', 'Digital Signal Processing Lab', 'Review Corporate Tax Laws', 'Business law brief', 'OS lab report', 'Marketing final exam', 'Drop box deadline for essay', 'Urban Planning site analysis', 'Professor office hours', 'SQL query optimization task', 'Creative Writing Workshop', 'Anatomy dissection lab', 'Blockchain smart contract project', 'Upload research data', 'AI model training project', 'European Politics Lecture', 'Hydrology lab', 'Peer review session', 'Journal Club Discussion', 'Public speaking workshop', 'American History Discussion', 'Quantum Physics lecture', 'Social Work field report', 'Tuition Fee Payment Due', 'Study cognitive psychology', 'Anatomy and Physiology Practical', 'Structural analysis project', 'React component task', 'Human resource strategy project', 'Computer vision project', 'Submit competition entry', 'Debate team meeting', 'Marketing Research Final', 'Social psychology class', 'Submit course feedback', 'Read Anthropology ethnographies', 'Discrete Math Midterm', 'Alumni Networking Call', 'Human Anatomy Lab Manual', 'Panel discussion', 'Mock interview meeting', 'Lab Partner Coordination', 'Python automation script', 'Criminal Justice Midterm', 'Finalize graduation paperwork', 'Geology Rock Identification Lab', 'AI research lab', 'Microeconomics midterm', 'Linux shell scripting task', 'Android app prototype', 'Art History Quiz', 'Practice SQL joins', 'Teaching Assistant Meeting', 'Environmental Science field trip', 'Renewable energy project', 'Dynamics homework', 'Archaeology dig report', 'Qualitative research paper', 'Software Requirement Spec document', 'Micro-finance lecture', 'Advanced robotics seminar', 'Upload Lab 5 to LMS', 'Study Shakespearean Sonnets', 'Art History visual analysis', 'Senior Design Prototype', 'Sustainable energy simulation', 'Review game theory models', 'Practice French Conjugation', 'Review cell signaling', 'Structural Analysis midterm', 'Review contract law cases', 'Public Speaking Practicum', 'Financial accounting ledger', 'Corporate finance seminar', 'Software Requirement Specification', 'FYP meeting', 'International relations workshop', 'Fluid Mechanics quiz', 'Public Health Case Analysis', 'Read Shakespearean play', 'Industrial Design Prototype', 'Biomedical Ethics Seminar', 'Mobile App Development Sprint', 'Thesis committee meeting', 'Civil Engineering Surveying Report', 'Math circle meeting', 'External Mentor Meeting', 'Read pharmacology textbook', 'Solve physics practice problems', 'Hand in physical portfolio', 'Portfolio Link Submission', 'Java programming lab', 'Macroeconomics lecture', 'Water purification project', 'Network Security Audit', 'Physical Chemistry Lab', 'Medieval History seminar', 'Upload portfolio to cloud', 'Data journalism workshop', 'Differential Equations Quiz', 'ML lab', 'Placement cell meeting', 'Orientation leader meeting', 'API integration assignment', 'Scholarship application cutoff', 'Linguistics Phonetics Assignment', 'Robotics kit assembly', 'Petroleum engineering lecture', 'Developmental Psychology Lecture', 'Education policy seminar', 'Calculus Problem Set 4', 'Urban Planning Design Sketch', 'Conference Abstract Submission', 'Quantum mechanics seminar', 'Multivariable calculus midterm', 'Macroeconomics Midterm', 'Sociology Quiz 3', 'Python Programming Homework', 'Fintech app project', 'Review discrete math', 'Radiology seminar', 'Geology field trip', 'Architecture studio project', 'Genetics midterm', 'Thesis Defense Mock', 'Alumni networking session', 'Study abroad info session', 'Signal processing test', 'Disaster management plan', 'Practice calculus integrals', 'Revise Marketing Mix Concepts', 'Neuroscience lecture', 'Autonomous vehicle simulation', 'Research assistant briefing', 'Study forensic science', 'Review Cognitive Psychology', 'Capstone design phase 1', 'Finalize budget sheet', 'Aerodynamics lab', 'Volunteer coordinator meeting', 'Inorganic Chemistry Quiz', 'Thesis defense rehearsal', 'Parasitology quiz', 'Molecular Biology Exam', 'Study for immunology', 'Academic advisor check-in', 'Language lab - Mandarin', 'Microeconomics Final', 'Analytical chemistry lab', 'Nursing clinical rotation', 'Review inorganic chem', 'Hand in lab notebook', 'Research Paper Final Draft', 'Clubs and societies meeting', 'Anatomy practical exam', 'Civil engineering drafting', 'Practice Accounting Ratios', 'Practice French dialogue', 'Review world geography', 'Neural Networks Implementation', 'Liaison meeting', "Dean's list reception", 'Accounting ledger exercise', 'Ethics in Tech seminar', 'Logic Gates Simulation', 'Geotechnical report project', 'Capstone Project Phase 1', 'Urban planning proposal', 'Logic and set theory class', 'Project manager sync', 'Anthropology Ethnography Draft', 'Revise Periodic Table Trends', 'Smart city project', 'Public policy proposal', 'Physical therapy lab', 'Music Theory Composition', 'Algorithm complexity assignment', 'Review Quantum Mechanics notes', 'Sociology midterm', 'Complete online certification', 'Register for Spring Semester', 'Structural Engineering Lecture', 'Archaeology seminar', 'Education Lesson Plan Portfolio', 'Volunteer coordination meeting', 'Education lesson plan project', 'Journalism Feature Article Draft', 'Peer Review Workshop', 'Virtual reality environment', 'Software Testing Tutorial', 'Upload audio transcript', 'Practice Python syntax', 'Late enrollment deadline', 'Practice Public Speaking', 'Data structures assignment', 'Fine Arts Exhibition Piece', 'Macroeconomics final', 'Submit travel reimbursement', 'Philosophy logic quiz', 'Microbiology culture log', 'Database project submission', 'Film production lab', 'Environmental law class', 'Materials Science Lecture', 'Statistics final', 'Circuit Analysis homework', 'Study for pathology', 'Email professor regarding extension', 'Operating Systems Lecture', 'Aeronautics lecture', 'IoT home automation', 'Memorize Spanish Vocabulary', 'Digital signal project', 'Finalize survey questions', 'Criminal Justice workshop', 'Register for spring exams', 'Confirm exam seat', 'Theatre Production Set Design', 'University press meeting', 'Food science lab', 'Practice Python logic', 'Read Modern History Chapter 5', 'AI chatbot development', 'Biomedical ethics seminar', 'Study Criminal Procedure']
    dates = ["today", "tonight", "tomorrow", "yesterday","the day after tomorrow", "day after","this Monday", "this Tuesday", "this Wednesday", "this Thursday","this Friday", "this Saturday", "this Sunday", "this weekend","next Monday", "next Tuesday", "next Wednesday", "next Thursday","next Friday", "next Saturday", "next Sunday", "next weekend", "next week","coming Monday", "coming Tuesday", "coming Wednesday", "coming Thursday","coming Friday", "coming Saturday", "coming Sunday", "coming weekend","upcoming Monday", "upcoming Tuesday", "upcoming Wednesday", "upcoming Thursday","upcoming Friday", "upcoming Saturday", "upcoming Sunday", "upcoming weekend","monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday","monday after next", "tuesday after next", "wednesday after next","thursday after next", "friday after next", "saturday after next", "sunday after next","end of the week", "end of the month", "next month","05/12", "07/04", "25/12", "01/01", "15/08", "20/03","20-03-2026", "25-12-2026", "07-04-2026", "01-01-2026", "15-08-2026","07/04/2026", "12/25/2026", "01/15/2026","Jan 15", "Feb 5", "Feb 28", "March 12", "April 7", "May 20","June 10", "July 15", "Aug 3", "Sep 18", "Oct 5", "Nov 12", "Dec 31","2nd Jan", "5th Feb", "12th March", "1st April", "20th June","15th July", "3rd Aug", "18th Sep", "5th Oct", "12th Nov", "31st Dec","1st Jan 2026", "2nd Feb 2026", "15th March 2026", "7th April 2026","20th June 2026", "15th July 2026", "31st Dec 2026", "15th Aug 2026","Feb 2nd 2026", "Oct 5th 2026", "Nov 12th 2026", "Jan 15th 2026","12th of October", "5th of February", "1st of January", "20th of March","15th of april","april 20","20 april"]
    times = ["5pm", "10:00 AM", "12:00", "midnight", "3:30", "18:00", "9:00", "4pm", "6:30","11:45", "2:00", "22:30", "08:00", "2pm", "14:00", "16:15", "9:45 PM", "7:00","00:00", "11:00 AM", "13:30", "21:00", "06:00", "3 PM", "10 AM", "4:45", "15:00","11:30", "1:15", "17:45", "09:15", "10:30 PM", "8:45", "12:30", "07:30", "23:00",  "12 AM", "12 PM", "1 AM", "1 PM", "2 AM", "2 PM", "3 AM", "4 AM", "4 PM", "5 AM", "6 AM","6 PM", "7 AM", "7 PM", "8 AM", "9 PM", "10 PM", "11 PM", "13:00", "14:30", "15:45", "16:00", "17:00", "18:15", "19:30", "20:00", "20:45", "21:15", "22:00", "23:30", "00:30", "01:15", "02:45", "03:00", "04:15", "05:30", "06:45", "07:00", "08:30", "09:15","morning", "afternoon", "evening", "night", "late night", "noon", "mid-morning", "early afternoon","late afternoon", "early evening", "late evening"]
    durations = ["1h","30m","10 minutes", "15 minutes", "20 minutes", "25 minutes", "30 minutes","35 minutes", "40 minutes", "45 minutes", "50 minutes", "55 minutes","1 hour", "2 hours", "3 hours", "4 hours", "5 hours","6 hours", "8 hours", "10 hours", "12 hours","1 hour 30 minutes", "1 hour 45 minutes", "2 hours 30 minutes","2 hours 15 minutes", "3 hours 30 minutes","30 mins", "45 mins", "20 mins", "15 mins", "10 mins","1 hr", "2 hrs", "3 hrs", "4 hrs", "5 hrs","1.5 hours", "2.5 hours", "3.5 hours","half an hour", "quarter of an hour", "an hour", "a few minutes",]
    priorities = ["high", "medium", "low","urgent", "critical", "immediate", "asap", "important","normal", "standard", "moderate", "average", "regular","minor", "trivial", "backburner", "background", "optional"]
    statutes = ["pending","ongoing","completed","in progress", "in-progress", "active","done", "finished", "resolved", "closed", "delivered", "fulfilled"]
    days = ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"]

    entity_values = {
        "day":days,
        # "task_name": task_names,
        # "date":      dates,
        # "time":      times,
        # "priority":  priorities,
        # "new_priority" : priorities,
        # "new_status":statutes,
        # "new_time":times,
        # "new_duration":durations,
        # "new_taskname":task_names,
        # "new_date":dates,
        # "status" : statutes,
        # "person":    collaborators,
        # "duration":  durations,
    }


    # ── generate ─────────────────────────────────────────────────────────────
    clean, flagged = generate(
        templates     = BEHAVIORAL_TEMPLATES,
        entity_values = entity_values,
        intent        = "behavioral",
        count         = 500,            # change to however many you need
    )

    print(f"\nGenerated : {len(clean)} clean  |  {len(flagged)} flagged")

    # ── quick preview ─────────────────────────────────────────────────────────
    for ex in clean[:3]:
        print(json.dumps(ex, indent=2))
    
    # ── save ──────────────────────────────────────────────────────────────────
    save_dataset(
        clean,
        flagged,
        clean_path   = r"/Users/hamidshehzad/FYP_CODE_20_March/FYP_CODE_INSHALLAH_FINAL --- VSCODE/DataSets/InshallahDataset/CompleteNerDataSet.json",
        flagged_path = r"DataSets\InshallahDataset\completeDataSetFlagged.json",
        append       = True,
    )

In [17]:
import json

def _normalize_entity_triplet(entity_item):
    """
    Supports both formats:
      1) {"start": 10, "end": 20, "entity": "date"}
      2) [10, 20, "date"] / (10, 20, "date")
    Returns (start, end, label) or None if invalid.
    """
    if isinstance(entity_item, dict):
        start = entity_item.get("start")
        end = entity_item.get("end")
        label = entity_item.get("entity") or entity_item.get("label")
    elif isinstance(entity_item, (list, tuple)) and len(entity_item) >= 3:
        start, end, label = entity_item[:3]
    else:
        return None

    if start is None or end is None or not label:
        return None

    try:
        return int(start), int(end), str(label)
    except (TypeError, ValueError):
        return None


def get_unique_entities(file_path):
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)

        # Using a set to ensure unique values
        unique_entities = set()
        total_count = 0

        for instance in data:
            # Check if entities key exists and is a list
            if "entities" in instance and isinstance(instance["entities"], list):
                for ent in instance["entities"]:
                    normalized = _normalize_entity_triplet(ent)
                    if normalized is None:
                        continue
                    _, _, label = normalized
                    unique_entities.add(label)
                    total_count += 1

        # Convert set to a sorted list for the final output
        entities_list = sorted(list(unique_entities))

        print(f"Total entity instances found: {total_count}")
        print(f"Unique entities ({len(entities_list)}): {entities_list}")

        return entities_list

    except FileNotFoundError:
        print("Error: The file was not found.")
    except json.JSONDecodeError:
        print("Error: Failed to decode JSON.")

# Replace 'dataset.json' with your actual filename
get_unique_entities(r'/Users/hamidshehzad/FYP_CODE_20_March/FYP_CODE_INSHALLAH_FINAL --- VSCODE/DataSets/InshallahDataset/show/dataset.json')

Total entity instances found: 7966
Unique entities (5): ['date', 'priority', 'status', 'task_name', 'time']


['date', 'priority', 'status', 'task_name', 'time']

In [ ]:
# ================================================================
# EVA — NER TRAINING TEMPLATES
# Format: (template_string, [entity_labels])
# Slot notation:
#   {task_name}  → task name
#   {date}       → date expression
#   {time}       → time expression
#   {priority}   → low / medium / high
#   {person}     → collaborator name
#   {duration}   → duration expression
#   {status}     → pending / ongoing / complete
#   {day}        → specific weekday
# ================================================================


# ================================================================
# ADD COMMAND
# ================================================================

ADD_TEMPLATES = [

    # ── COMBINATION 1 — task only ────────────────────────────────
    ("add {task_name}", ["task_name"]),
    ("add task {task_name}", ["task_name"]),
    ("schedule {task_name}", ["task_name"]),
    ("create {task_name}", ["task_name"]),
    ("set {task_name}", ["task_name"]),
    ("add a task called {task_name}", ["task_name"]),
    ("add a new task for {task_name}", ["task_name"]),
    ("create a task named {task_name}", ["task_name"]),
    ("schedule a task called {task_name}", ["task_name"]),
    ("can you add {task_name}", ["task_name"]),
    ("I need to add {task_name}", ["task_name"]),
    ("please add {task_name}", ["task_name"]),
    ("I want to create {task_name}", ["task_name"]),
    ("set up {task_name} for me", ["task_name"]),
    ("I have to do {task_name}", ["task_name"]),
    ("could u please add {task_name}",["task_name"]),

    # ── COMBINATION 2 — task + date ──────────────────────────────
    ("add {task_name} {date}", ["task_name", "date"]),
    ("schedule {task_name} {date}", ["task_name", "date"]),
    ("add {task_name} on {date}", ["task_name", "date"]),
    ("add {task_name} for {date}", ["task_name", "date"]),
    ("add {task_name} by {date}", ["task_name", "date"]),
    ("schedule {task_name} on {date}", ["task_name", "date"]),
    ("create {task_name} for {date}", ["task_name", "date"]),
    ("set {task_name} on {date}", ["task_name", "date"]),
    ("add {task_name} this {date}", ["task_name", "date"]),
    ("add {task_name} next {date}", ["task_name", "date"]),
    ("on {date} add {task_name}", ["task_name", "date"]),
    ("add {task_name} and set it for {date}", ["task_name", "date"]),
    ("can you schedule {task_name} on {date}", ["task_name", "date"]),
    ("I want to add {task_name} on {date}", ["task_name", "date"]),
    ("please schedule {task_name} for {date}", ["task_name", "date"]),
    ("I have {task_name} on {date}", ["task_name", "date"]),
    ("remind me about {task_name} on {date}", ["task_name", "date"]),
    ("create a task for {task_name} on {date}", ["task_name", "date"]),

    # ── COMBINATION 3 — task + time ──────────────────────────────
    ("add {task_name} {time}", ["task_name", "time"]),
    ("schedule {task_name} {time}", ["task_name", "time"]),
    ("add {task_name} at {time}", ["task_name", "time"]),
    ("add {task_name} for {time}", ["task_name", "time"]),
    ("add {task_name} by {time}", ["task_name", "time"]),
    ("add {task_name} around {time}", ["task_name", "time"]),
    ("schedule {task_name} at {time}", ["task_name", "time"]),
    ("create {task_name} at {time}", ["task_name", "time"]),
    ("set {task_name} for {time}", ["task_name", "time"]),
    ("add {task_name} at around {time}", ["task_name", "time"]),
    ("at {time} add {task_name}", ["task_name", "time"]),
    ("add {task_name} and set time to {time}", ["task_name", "time"]),
    ("can you add {task_name} at {time}", ["task_name", "time"]),
    ("I want {task_name} scheduled at {time}", ["task_name", "time"]),
    ("I have {task_name} at {time}", ["task_name", "time"]),
    ("put {task_name} at {time}", ["task_name", "time"]),

    # ── COMBINATION 4 — task + priority ──────────────────────────
    ("add {task_name} {priority}", ["task_name", "priority"]),
    ("schedule {task_name} {priority} priority", ["task_name", "priority"]),
    ("add {task_name} with {priority} priority", ["task_name", "priority"]),
    ("add {task_name} marked as {priority}", ["task_name", "priority"]),
    ("add {task_name} as {priority} priority", ["task_name", "priority"]),
    ("add {task_name} with priority {priority}", ["task_name", "priority"]),
    ("create {task_name} with {priority} priority", ["task_name", "priority"]),
    ("schedule {task_name} with {priority} priority", ["task_name", "priority"]),
    ("set {task_name} as {priority}", ["task_name", "priority"]),
    ("add {task_name} priority {priority}", ["task_name", "priority"]),
    ("add {task_name} and mark it {priority}", ["task_name", "priority"]),
    ("can you add {task_name} with {priority} priority", ["task_name", "priority"]),
    ("I want {task_name} with {priority} priority", ["task_name", "priority"]),
    ("create a {priority} priority task for {task_name}", ["task_name", "priority"]),
    ("I need to add {task_name} it is {priority} priority", ["task_name", "priority"]),

    # ── COMBINATION 5 — task + person ────────────────────────────
    ("add {task_name} {person}", ["task_name", "person"]),
    ("schedule {task_name} {person}", ["task_name", "person"]),
    ("add {task_name} with {person}", ["task_name", "person"]),
    ("add {task_name} alongside {person}", ["task_name", "person"]),
    ("add {task_name} together with {person}", ["task_name", "person"]),
    ("schedule {task_name} with {person}", ["task_name", "person"]),
    ("create {task_name} with {person}", ["task_name", "person"]),
    ("add {task_name} and include {person}", ["task_name", "person"]),
    ("add {task_name} and add {person}", ["task_name", "person"]),
    ("with {person} add {task_name}", ["task_name", "person"]),
    ("can you add {task_name} with {person}", ["task_name", "person"]),
    ("I want to add {task_name} with {person}", ["task_name", "person"]),
    ("add {task_name} and also add {person} to it", ["task_name", "person"]),
    ("schedule {task_name} and invite {person}", ["task_name", "person"]),
    ("create {task_name} for me and {person}", ["task_name", "person"]),

    # ── COMBINATION 6 — task + duration ──────────────────────────
    ("add {task_name} {duration}", ["task_name", "duration"]),
    ("schedule {task_name} {duration}", ["task_name", "duration"]),
    ("add {task_name} for {duration}", ["task_name", "duration"]),
    ("add {task_name} lasting {duration}", ["task_name", "duration"]),
    ("add {task_name} takes {duration}", ["task_name", "duration"]),
    ("schedule {task_name} for {duration}", ["task_name", "duration"]),
    ("create {task_name} of {duration}", ["task_name", "duration"]),
    ("add {task_name} duration {duration}", ["task_name", "duration"]),
    ("set {task_name} for {duration}", ["task_name", "duration"]),
    ("add {task_name} and set duration to {duration}", ["task_name", "duration"]),
    ("can you add {task_name} for {duration}", ["task_name", "duration"]),
    ("I want to add {task_name} lasting {duration}", ["task_name", "duration"]),
    ("create {task_name} it will take {duration}", ["task_name", "duration"]),
    ("schedule {task_name} duration is {duration}", ["task_name", "duration"]),

    # ── COMBINATION 7 — task + date + time ───────────────────────
    ("add {task_name} {date} {time}", ["task_name", "date", "time"]),
    ("schedule {task_name} {date} {time}", ["task_name", "date", "time"]),
    ("add {task_name} on {date} at {time}", ["task_name", "date", "time"]),
    ("add {task_name} for {date} at {time}", ["task_name", "date", "time"]),
    ("schedule {task_name} on {date} at {time}", ["task_name", "date", "time"]),
    ("create {task_name} on {date} at {time}", ["task_name", "date", "time"]),
    ("set {task_name} on {date} at {time}", ["task_name", "date", "time"]),
    ("add {task_name} on {date} for {time}", ["task_name", "date", "time"]),
    ("add {task_name} by {date} at {time}", ["task_name", "date", "time"]),
    ("on {date} at {time} add {task_name}", ["task_name", "date", "time"]),
    ("at {time} on {date} add {task_name}", ["task_name", "date", "time"]),
    ("add {task_name} at {time} on {date}", ["task_name", "date", "time"]),
    ("can you schedule {task_name} on {date} at {time}", ["task_name", "date", "time"]),
    ("I need {task_name} on {date} at {time}", ["task_name", "date", "time"]),
    ("please add {task_name} at {time} on {date}", ["task_name", "date", "time"]),
    ("I have {task_name} on {date} at {time}", ["task_name", "date", "time"]),
    ("remind me of {task_name} on {date} at {time}", ["task_name", "date", "time"]),

    # ── COMBINATION 8 — task + date + priority ───────────────────
    ("add {task_name} {date} {priority}", ["task_name", "date", "priority"]),
    ("schedule {task_name} {date} {priority} priority", ["task_name", "date", "priority"]),
    ("add {task_name} on {date} with {priority} priority", ["task_name", "date", "priority"]),
    ("add {task_name} for {date} with {priority} priority", ["task_name", "date", "priority"]),
    ("add {task_name} on {date} as {priority}", ["task_name", "date", "priority"]),
    ("add {task_name} on {date} marked {priority}", ["task_name", "date", "priority"]),
    ("schedule {task_name} on {date} with {priority} priority", ["task_name", "date", "priority"]),
    ("create {task_name} on {date} priority {priority}", ["task_name", "date", "priority"]),
    ("on {date} add {task_name} with {priority} priority", ["task_name", "date", "priority"]),
    ("add {task_name} with {priority} priority on {date}", ["task_name", "date", "priority"]),
    ("can you add {task_name} on {date} with {priority} priority", ["task_name", "date", "priority"]),
    ("I want to schedule {task_name} on {date} it is {priority}", ["task_name", "date", "priority"]),
    ("could u add {task_name} on {date} it is {priority}", ["task_name", "date", "priority"]),
    ("could u remind me about {task_name} on {date} it is {priority}", ["task_name", "date", "priority"]),

    # ── COMBINATION 9 — task + date + person ─────────────────────
    ("add {task_name} {date} {person}", ["task_name", "date", "person"]),
    ("schedule {task_name} {date} {person}", ["task_name", "date", "person"]),
    ("add {task_name} on {date} with {person}", ["task_name", "date", "person"]),
    ("add {task_name} for {date} with {person}", ["task_name", "date", "person"]),
    ("schedule {task_name} on {date} with {person}", ["task_name", "date", "person"]),
    ("add {task_name} on {date} and include {person}", ["task_name", "date", "person"]),
    ("create {task_name} on {date} alongside {person}", ["task_name", "date", "person"]),
    ("add {task_name} with {person} on {date}", ["task_name", "date", "person"]),
    ("with {person} add {task_name} on {date}", ["task_name", "date", "person"]),
    ("on {date} add {task_name} with {person}", ["task_name", "date", "person"]),
    ("can you add {task_name} on {date} with {person}", ["task_name", "date", "person"]),
    ("I want {task_name} on {date} with {person}", ["task_name", "date", "person"]),
    ("schedule {task_name} with {person} for {date}", ["task_name", "date", "person"]),

    # ── COMBINATION 10 — task + date + duration ──────────────────
    ("add {task_name} {date} {duration}", ["task_name", "date", "duration"]),
    ("schedule {task_name} {date} {duration}", ["task_name", "date", "duration"]),
    ("add {task_name} on {date} for {duration}", ["task_name", "date", "duration"]),
    ("add {task_name} on {date} lasting {duration}", ["task_name", "date", "duration"]),
    ("schedule {task_name} on {date} for {duration}", ["task_name", "date", "duration"]),
    ("create {task_name} on {date} of {duration}", ["task_name", "date", "duration"]),
    ("add {task_name} for {date} lasting {duration}", ["task_name", "date", "duration"]),
    ("on {date} add {task_name} for {duration}", ["task_name", "date", "duration"]),
    ("add {task_name} lasting {duration} on {date}", ["task_name", "date", "duration"]),
    ("can you add {task_name} on {date} for {duration}", ["task_name", "date", "duration"]),
    ("I want to add {task_name} on {date} it takes {duration}", ["task_name", "date", "duration"]),

    # ── COMBINATION 11 — task + date + time + priority ───────────
    ("add {task_name} {date} {time} {priority}", ["task_name", "date", "time", "priority"]),
    ("schedule {task_name} {date} {time} {priority} priority", ["task_name", "date", "time", "priority"]),
    ("add {task_name} on {date} at {time} with {priority} priority", ["task_name", "date", "time", "priority"]),
    ("add {task_name} on {date} at {time} as {priority}", ["task_name", "date", "time", "priority"]),
    ("schedule {task_name} on {date} at {time} with {priority} priority", ["task_name", "date", "time", "priority"]),
    ("create {task_name} on {date} at {time} priority {priority}", ["task_name", "date", "time", "priority"]),
    ("add {task_name} with {priority} priority on {date} at {time}", ["task_name", "date", "time", "priority"]),
    ("on {date} at {time} add {task_name} with {priority} priority", ["task_name", "date", "time", "priority"]),
    ("add {task_name} at {time} on {date} with {priority} priority", ["task_name", "date", "time", "priority"]),
    ("can you add {task_name} on {date} at {time} with {priority} priority", ["task_name", "date", "time", "priority"]),
    ("I have {task_name} on {date} at {time} it is {priority}", ["task_name", "date", "time", "priority"]),

    # ── COMBINATION 12 — task + date + time + person ─────────────
    ("add {task_name} {date} {time} {person}", ["task_name", "date", "time", "person"]),
    ("schedule {task_name} {date} {time} {person}", ["task_name", "date", "time", "person"]),
    ("add {task_name} on {date} at {time} with {person}", ["task_name", "date", "time", "person"]),
    ("add {task_name} on {date} at {time} alongside {person}", ["task_name", "date", "time", "person"]),
    ("schedule {task_name} on {date} at {time} with {person}", ["task_name", "date", "time", "person"]),
    ("add {task_name} with {person} on {date} at {time}", ["task_name", "date", "time", "person"]),
    ("create {task_name} on {date} at {time} and add {person}", ["task_name", "date", "time", "person"]),
    ("with {person} add {task_name} on {date} at {time}", ["task_name", "date", "time", "person"]),
    ("add {task_name} at {time} on {date} with {person}", ["task_name", "date", "time", "person"]),
    ("on {date} at {time} add {task_name} with {person}", ["task_name", "date", "time", "person"]),
    ("can you schedule {task_name} on {date} at {time} with {person}", ["task_name", "date", "time", "person"]),
    ("I need {task_name} on {date} at {time} with {person}", ["task_name", "date", "time", "person"]),

    # ── COMBINATION 13 — task + date + time + duration ───────────
    ("add {task_name} {date} {time} {duration}", ["task_name", "date", "time", "duration"]),
    ("schedule {task_name} {date} {time} {duration}", ["task_name", "date", "time", "duration"]),
    ("add {task_name} on {date} at {time} for {duration}", ["task_name", "date", "time", "duration"]),
    ("add {task_name} on {date} at {time} lasting {duration}", ["task_name", "date", "time", "duration"]),
    ("schedule {task_name} on {date} at {time} for {duration}", ["task_name", "date", "time", "duration"]),
    ("create {task_name} on {date} at {time} of {duration}", ["task_name", "date", "time", "duration"]),
    ("on {date} at {time} add {task_name} for {duration}", ["task_name", "date", "time", "duration"]),
    ("add {task_name} lasting {duration} on {date} at {time}", ["task_name", "date", "time", "duration"]),
    ("add {task_name} for {duration} on {date} at {time}", ["task_name", "date", "time", "duration"]),
    ("can you add {task_name} on {date} at {time} for {duration}", ["task_name", "date", "time", "duration"]),
    ("I have {task_name} on {date} at {time} for {duration}", ["task_name", "date", "time", "duration"]),

    # ── COMBINATION 14 — task + date + time + priority + person ──
    ("add {task_name} {date} {time} {priority} {person}", ["task_name", "date", "time", "priority", "person"]),
    ("add {task_name} on {date} at {time} with {priority} priority with {person}", ["task_name", "date", "time", "priority", "person"]),
    ("add {task_name} on {date} at {time} as {priority} with {person}", ["task_name", "date", "time", "priority", "person"]),
    ("schedule {task_name} on {date} at {time} with {priority} priority alongside {person}", ["task_name", "date", "time", "priority", "person"]),
    ("add {task_name} with {person} on {date} at {time} with {priority} priority", ["task_name", "date", "time", "priority", "person"]),
    ("with {person} add {task_name} on {date} at {time} with {priority} priority", ["task_name", "date", "time", "priority", "person"]),
    ("add {task_name} at {time} on {date} with {person} and {priority} priority", ["task_name", "date", "time", "priority", "person"]),
    ("can you add {task_name} on {date} at {time} with {priority} priority with {person}", ["task_name", "date", "time", "priority", "person"]),
    ("I need {task_name} on {date} at {time} it is {priority} add {person} too", ["task_name", "date", "time", "priority", "person"]),

    # ── COMBINATION 15 — task + date + time + priority + duration ─
    ("add {task_name} {date} {time} {priority} {duration}", ["task_name", "date", "time", "priority", "duration"]),
    ("add {task_name} on {date} at {time} with {priority} priority for {duration}", ["task_name", "date", "time", "priority", "duration"]),
    ("add {task_name} on {date} at {time} as {priority} lasting {duration}", ["task_name", "date", "time", "priority", "duration"]),
    ("schedule {task_name} on {date} at {time} with {priority} priority for {duration}", ["task_name", "date", "time", "priority", "duration"]),
    ("add {task_name} for {duration} on {date} at {time} with {priority} priority", ["task_name", "date", "time", "priority", "duration"]),
    ("can you add {task_name} on {date} at {time} with {priority} priority for {duration}", ["task_name", "date", "time", "priority", "duration"]),
    ("I have {task_name} on {date} at {time} it is {priority} and lasts {duration}", ["task_name", "date", "time", "priority", "duration"]),

    # ── COMBINATION 16 — task + date + time + priority + person + duration
    ("add {task_name} {date} {time} {priority} {person} {duration}", ["task_name", "date", "time", "priority", "person", "duration"]),
    ("add {task_name} on {date} at {time} with {priority} priority with {person} for {duration}", ["task_name", "date", "time", "priority", "person", "duration"]),
    ("add {task_name} on {date} at {time} as {priority} alongside {person} lasting {duration}", ["task_name", "date", "time", "priority", "person", "duration"]),
    ("schedule {task_name} on {date} at {time} with {priority} priority with {person} for {duration}", ["task_name", "date", "time", "priority", "person", "duration"]),
    ("with {person} add {task_name} on {date} at {time} with {priority} priority for {duration}", ["task_name", "date", "time", "priority", "person", "duration"]),
    ("add {task_name} for {duration} on {date} at {time} with {priority} priority and {person}", ["task_name", "date", "time", "priority", "person", "duration"]),
    ("I need {task_name} on {date} at {time} {priority} priority with {person} for {duration}", ["task_name", "date", "time", "priority", "person", "duration"]),
]


# ================================================================
# SHOW COMMAND
# ================================================================

    


# ================================================================
# DELETE COMMAND
# ================================================================

DELETE_TEMPLATES = [

    # ── DELETE SPECIFIC — task only ──────────────────────────────
    ("delete {task_name}", ["task_name"]),
    ("remove {task_name}", ["task_name"]),
    ("cancel {task_name}", ["task_name"]),
    ("delete task {task_name}", ["task_name"]),
    ("remove task {task_name}", ["task_name"]),
    ("delete the {task_name}", ["task_name"]),
    ("remove the {task_name}", ["task_name"]),
    ("cancel the {task_name}", ["task_name"]),
    ("delete a task called {task_name}", ["task_name"]),
    ("can you delete {task_name}", ["task_name"]),
    ("I want to delete {task_name}", ["task_name"]),
    ("please remove {task_name}", ["task_name"]),
    ("get rid of {task_name}", ["task_name"]),
    ("I no longer need {task_name}", ["task_name"]),
    ("drop {task_name}", ["task_name"]),

    # ── DELETE SPECIFIC — time only ──────────────────────────────
    ("delete task {time}", ["time"]),
    ("remove task {time}", ["time"]),
    ("delete task at {time}", ["time"]),
    ("remove task at {time}", ["time"]),
    ("delete the task at {time}", ["time"]),
    ("remove task scheduled at {time}", ["time"]),
    ("cancel task at {time}", ["time"]),
    ("at {time} delete task", ["time"]),
    ("can you delete the task at {time}", ["time"]),
    ("I want to remove the task at {time}", ["time"]),

    # ── DELETE SPECIFIC — task + date ────────────────────────────
    ("delete {task_name} {date}", ["task_name", "date"]),
    ("remove {task_name} {date}", ["task_name", "date"]),
    ("delete {task_name} on {date}", ["task_name", "date"]),
    ("remove {task_name} on {date}", ["task_name", "date"]),
    ("delete {task_name} for {date}", ["task_name", "date"]),
    ("cancel {task_name} on {date}", ["task_name", "date"]),
    ("remove {task_name} scheduled on {date}", ["task_name", "date"]),
    ("delete the {task_name} on {date}", ["task_name", "date"]),
    ("on {date} delete {task_name}", ["task_name", "date"]),
    ("can you delete {task_name} on {date}", ["task_name", "date"]),
    ("I want to remove {task_name} on {date}", ["task_name", "date"]),
    ("please delete {task_name} for {date}", ["task_name", "date"]),
    ("get rid of {task_name} on {date}", ["task_name", "date"]),

    # ── DELETE SPECIFIC — task + time ────────────────────────────
    ("delete {task_name} {time}", ["task_name", "time"]),
    ("remove {task_name} {time}", ["task_name", "time"]),
    ("delete {task_name} at {time}", ["task_name", "time"]),
    ("remove {task_name} at {time}", ["task_name", "time"]),
    ("cancel {task_name} at {time}", ["task_name", "time"]),
    ("delete the {task_name} at {time}", ["task_name", "time"]),
    ("remove {task_name} scheduled at {time}", ["task_name", "time"]),
    ("at {time} delete {task_name}", ["task_name", "time"]),
    ("can you delete {task_name} at {time}", ["task_name", "time"]),
    ("I want to remove {task_name} at {time}", ["task_name", "time"]),
    ("please remove {task_name} at {time}", ["task_name", "time"]),

    # ── DELETE BULK — date only ──────────────────────────────────
    ("delete tasks {date}", ["date"]),
    ("remove tasks {date}", ["date"]),
    ("delete tasks on {date}", ["date"]),
    ("remove tasks on {date}", ["date"]),
    ("delete all tasks on {date}", ["date"]),
    ("remove all tasks on {date}", ["date"]),
    ("cancel tasks on {date}", ["date"]),
    ("delete tasks for {date}", ["date"]),
    ("on {date} delete tasks", ["date"]),
    ("can you delete tasks on {date}", ["date"]),
    ("I want to remove all tasks on {date}", ["date"]),
    ("clear my schedule for {date}", ["date"]),

    # ── DELETE BULK — status only ────────────────────────────────
    ("delete {status} tasks", ["status"]),
    ("remove {status} tasks", ["status"]),
    ("delete all {status} tasks", ["status"]),
    ("remove all {status} tasks", ["status"]),
    ("delete tasks that are {status}", ["status"]),
    ("cancel {status} tasks", ["status"]),
    ("can you delete {status} tasks", ["status"]),
    ("I want to remove {status} tasks", ["status"]),
    ("clear all {status} tasks", ["status"]),
    ("get rid of {status} tasks", ["status"]),

    # ── DELETE BULK — priority only ──────────────────────────────
    ("delete {priority} tasks", ["priority"]),
    ("remove {priority} priority tasks", ["priority"]),
    ("delete all {priority} priority tasks", ["priority"]),
    ("remove all {priority} tasks", ["priority"]),
    ("delete tasks with {priority} priority", ["priority"]),
    ("remove {priority} priority tasks", ["priority"]),
    ("cancel {priority} priority tasks", ["priority"]),
    ("can you delete {priority} priority tasks", ["priority"]),
    ("I want to remove {priority} tasks", ["priority"]),
    ("clear {priority} priority tasks", ["priority"]),

    # ── DELETE BULK — date + status ──────────────────────────────
    ("delete {status} tasks {date}", ["status", "date"]),
    ("remove {status} tasks {date}", ["status", "date"]),
    ("delete {status} tasks on {date}", ["status", "date"]),
    ("remove {status} tasks on {date}", ["status", "date"]),
    ("delete all {status} tasks on {date}", ["status", "date"]),
    ("cancel {status} tasks on {date}", ["status", "date"]),
    ("remove tasks that are {status} on {date}", ["status", "date"]),
    ("on {date} delete {status} tasks", ["status", "date"]),
    ("can you delete {status} tasks on {date}", ["status", "date"]),
    ("I want to remove {status} tasks on {date}", ["status", "date"]),

    # ── DELETE BULK — date + priority ────────────────────────────
    ("delete {priority} tasks {date}", ["priority", "date"]),
    ("remove {priority} priority tasks {date}", ["priority", "date"]),
    ("delete {priority} priority tasks on {date}", ["priority", "date"]),
    ("remove {priority} tasks on {date}", ["priority", "date"]),
    ("delete all {priority} priority tasks on {date}", ["priority", "date"]),
    ("cancel {priority} priority tasks on {date}", ["priority", "date"]),
    ("on {date} delete {priority} priority tasks", ["priority", "date"]),
    ("can you delete {priority} tasks on {date}", ["priority", "date"]),
    ("I want to remove {priority} priority tasks on {date}", ["priority", "date"]),

    # ── DELETE BULK — status + priority ──────────────────────────
    ("delete {status} {priority} tasks", ["status", "priority"]),
    ("remove {status} {priority} priority tasks", ["status", "priority"]),
    ("delete all {status} {priority} priority tasks", ["status", "priority"]),
    ("remove {status} tasks with {priority} priority", ["status", "priority"]),
    ("delete {priority} priority {status} tasks", ["status", "priority"]),
    ("cancel {status} {priority} tasks", ["status", "priority"]),
    ("can you delete {status} {priority} priority tasks", ["status", "priority"]),
    ("I want to remove {priority} priority {status} tasks", ["status", "priority"]),
    ("clear {status} {priority} tasks", ["status", "priority"]),

    # ── DELETE BULK — date + status + priority ───────────────────
    ("delete {status} {priority} tasks {date}", ["status", "priority", "date"]),
    ("remove {date} {status} {priority} tasks", ["status", "priority", "date"]),
    ("delete {status} {priority} priority tasks on {date}", ["status", "priority", "date"]),
    ("remove all {status} {priority} tasks on {date}", ["status", "priority", "date"]),
    ("delete {priority} priority {status} tasks on {date}", ["status", "priority", "date"]),
    ("cancel {status} {priority} tasks on {date}", ["status", "priority", "date"]),
    ("on {date} delete {status} {priority} priority tasks", ["status", "priority", "date"]),
    ("can you delete {status} {priority} priority tasks on {date}", ["status", "priority", "date"]),
    ("I want to remove {priority} {status} tasks on {date}", ["status", "priority", "date"]),
    ("clear {status} {priority} tasks on {date}", ["status", "priority", "date"]),
]


# ================================================================
# UPDATE COMMAND
# (identifier entities only — update fields are rule-based)
# ================================================================

# ================================================================
# EVA — UPDATE COMMAND NER TEMPLATES
# Entities:
#   Identifiers : task_name, date, time, priority, status
#   New values  : new_date, new_time, new_priority,
#                 new_status, new_duration, new_taskname
# ================================================================

UPDATE_TEMPLATES = [

    # ================================================================
    # SINGLE FIELD — new_priority only
    # ================================================================

    # task + new_priority
    ("change {task_name} priority to {new_priority}", ["task_name", "new_priority"]),
    ("update {task_name} priority to {new_priority}", ["task_name", "new_priority"]),
    ("set {task_name} priority to {new_priority}", ["task_name", "new_priority"]),
    ("mark {task_name} as {new_priority} priority", ["task_name", "new_priority"]),
    ("make {task_name} {new_priority} priority", ["task_name", "new_priority"]),
    ("update the priority of {task_name} to {new_priority}", ["task_name", "new_priority"]),
    ("change the priority of {task_name} to {new_priority}", ["task_name", "new_priority"]),
    ("{task_name} change priority to {new_priority}", ["task_name", "new_priority"]),
    ("I want to change {task_name} priority to {new_priority}", ["task_name", "new_priority"]),
    ("can you update {task_name} to {new_priority} priority", ["task_name", "new_priority"]),
    ("please set {task_name} as {new_priority}", ["task_name", "new_priority"]),
    ("bump {task_name} to {new_priority} priority", ["task_name", "new_priority"]),

    # date + new_priority
    ("change task on {date} priority to {new_priority}", ["date", "new_priority"]),
    ("update task on {date} to {new_priority} priority", ["date", "new_priority"]),
    ("set task on {date} as {new_priority}", ["date", "new_priority"]),
    ("mark task on {date} as {new_priority} priority", ["date", "new_priority"]),

    # old_priority + new_priority
    ("change {priority} priority task to {new_priority}", ["priority", "new_priority"]),
    ("update {priority} task to {new_priority} priority", ["priority", "new_priority"]),
    ("change priority from {priority} to {new_priority}", ["priority", "new_priority"]),
    ("update priority from {priority} to {new_priority}", ["priority", "new_priority"]),

    # task + date + new_priority
    ("change {task_name} on {date} priority to {new_priority}", ["task_name", "date", "new_priority"]),
    ("update {task_name} on {date} to {new_priority} priority", ["task_name", "date", "new_priority"]),
    ("set {task_name} on {date} as {new_priority} priority", ["task_name", "date", "new_priority"]),
    ("mark {task_name} on {date} as {new_priority}", ["task_name", "date", "new_priority"]),

    # ================================================================
    # SINGLE FIELD — new_status only
    # ================================================================

    # task + new_status
    ("mark {task_name} as {new_status}", ["task_name", "new_status"]),
    ("update {task_name} status to {new_status}", ["task_name", "new_status"]),
    ("change {task_name} status to {new_status}", ["task_name", "new_status"]),
    ("set {task_name} as {new_status}", ["task_name", "new_status"]),
    ("change status of {task_name} to {new_status}", ["task_name", "new_status"]),
    ("update the status of {task_name} to {new_status}", ["task_name", "new_status"]),
    ("{task_name} mark as {new_status}", ["task_name", "new_status"]),
    ("I want to mark {task_name} as {new_status}", ["task_name", "new_status"]),
    ("can you update {task_name} to {new_status}", ["task_name", "new_status"]),
    ("please mark {task_name} as {new_status}", ["task_name", "new_status"]),
    ("set {task_name} status as {new_status}", ["task_name", "new_status"]),
    ("change {task_name} to {new_status}", ["task_name", "new_status"]),

    # old_status + new_status
    ("change {status} task to {new_status}", ["status", "new_status"]),
    ("update {status} task status to {new_status}", ["status", "new_status"]),
    ("change status from {status} to {new_status}", ["status", "new_status"]),
    ("update status from {status} to {new_status}", ["status", "new_status"]),
    ("mark {status} task as {new_status}", ["status", "new_status"]),

    # date + new_status
    ("mark task on {date} as {new_status}", ["date", "new_status"]),
    ("update task on {date} status to {new_status}", ["date", "new_status"]),
    ("change task on {date} to {new_status}", ["date", "new_status"]),
    ("set task on {date} as {new_status}", ["date", "new_status"]),

    # task + date + new_status
    ("mark {task_name} on {date} as {new_status}", ["task_name", "date", "new_status"]),
    ("update {task_name} on {date} status to {new_status}", ["task_name", "date", "new_status"]),
    ("change {task_name} on {date} to {new_status}", ["task_name", "date", "new_status"]),
    ("set {task_name} on {date} as {new_status}", ["task_name", "date", "new_status"]),

    # ================================================================
    # SINGLE FIELD — new_date only
    # ================================================================

    # task + new_date
    ("reschedule {task_name} to {new_date}", ["task_name", "new_date"]),
    ("move {task_name} to {new_date}", ["task_name", "new_date"]),
    ("change {task_name} date to {new_date}", ["task_name", "new_date"]),
    ("update {task_name} date to {new_date}", ["task_name", "new_date"]),
    ("shift {task_name} to {new_date}", ["task_name", "new_date"]),
    ("change the date of {task_name} to {new_date}", ["task_name", "new_date"]),
    ("update the date of {task_name} to {new_date}", ["task_name", "new_date"]),
    ("postpone {task_name} to {new_date}", ["task_name", "new_date"]),
    ("push {task_name} to {new_date}", ["task_name", "new_date"]),
    ("I want to move {task_name} to {new_date}", ["task_name", "new_date"]),
    ("can you reschedule {task_name} to {new_date}", ["task_name", "new_date"]),
    ("please move {task_name} to {new_date}", ["task_name", "new_date"]),

    # old_date + new_date
    ("move task from {date} to {new_date}", ["date", "new_date"]),
    ("reschedule task from {date} to {new_date}", ["date", "new_date"]),
    ("change task on {date} to {new_date}", ["date", "new_date"]),
    ("update task on {date} to {new_date}", ["date", "new_date"]),
    ("shift task from {date} to {new_date}", ["date", "new_date"]),
    ("push task from {date} to {new_date}", ["date", "new_date"]),
    ("postpone task on {date} to {new_date}", ["date", "new_date"]),

    # task + old_date + new_date
    ("move {task_name} from {date} to {new_date}", ["task_name", "date", "new_date"]),
    ("reschedule {task_name} from {date} to {new_date}", ["task_name", "date", "new_date"]),
    ("change {task_name} on {date} to {new_date}", ["task_name", "date", "new_date"]),
    ("update {task_name} on {date} to {new_date}", ["task_name", "date", "new_date"]),
    ("shift {task_name} from {date} to {new_date}", ["task_name", "date", "new_date"]),
    ("push {task_name} on {date} to {new_date}", ["task_name", "date", "new_date"]),
    ("postpone {task_name} from {date} to {new_date}", ["task_name", "date", "new_date"]),
    ("I want to move {task_name} from {date} to {new_date}", ["task_name", "date", "new_date"]),

    # ================================================================
    # SINGLE FIELD — new_time only
    # ================================================================

    # task + new_time
    ("change {task_name} time to {new_time}", ["task_name", "new_time"]),
    ("update {task_name} time to {new_time}", ["task_name", "new_time"]),
    ("set {task_name} to {new_time}", ["task_name", "new_time"]),
    ("move {task_name} to {new_time}", ["task_name", "new_time"]),
    ("change the time of {task_name} to {new_time}", ["task_name", "new_time"]),
    ("update the time of {task_name} to {new_time}", ["task_name", "new_time"]),
    ("reschedule {task_name} to {new_time}", ["task_name", "new_time"]),
    ("shift {task_name} to {new_time}", ["task_name", "new_time"]),
    ("I want to change {task_name} to {new_time}", ["task_name", "new_time"]),
    ("can you update {task_name} time to {new_time}", ["task_name", "new_time"]),
    ("please change {task_name} to {new_time}", ["task_name", "new_time"]),

    # old_time + new_time
    ("change task at {time} to {new_time}", ["time", "new_time"]),
    ("update task at {time} to {new_time}", ["time", "new_time"]),
    ("move task from {time} to {new_time}", ["time", "new_time"]),
    ("reschedule task at {time} to {new_time}", ["time", "new_time"]),
    ("shift task from {time} to {new_time}", ["time", "new_time"]),
    ("change time from {time} to {new_time}", ["time", "new_time"]),
    ("update time from {time} to {new_time}", ["time", "new_time"]),

    # task + old_time + new_time
    ("change {task_name} from {time} to {new_time}", ["task_name", "time", "new_time"]),
    ("move {task_name} from {time} to {new_time}", ["task_name", "time", "new_time"]),
    ("reschedule {task_name} from {time} to {new_time}", ["task_name", "time", "new_time"]),
    ("update {task_name} at {time} to {new_time}", ["task_name", "time", "new_time"]),
    ("change {task_name} time from {time} to {new_time}", ["task_name", "time", "new_time"]),
    ("shift {task_name} from {time} to {new_time}", ["task_name", "time", "new_time"]),

    # date + new_time
    ("change task on {date} time to {new_time}", ["date", "new_time"]),
    ("update task on {date} to {new_time}", ["date", "new_time"]),
    ("move task on {date} to {new_time}", ["date", "new_time"]),
    ("set task on {date} to {new_time}", ["date", "new_time"]),

    # task + date + new_time
    ("change {task_name} on {date} to {new_time}", ["task_name", "date", "new_time"]),
    ("update {task_name} on {date} time to {new_time}", ["task_name", "date", "new_time"]),
    ("move {task_name} on {date} to {new_time}", ["task_name", "date", "new_time"]),
    ("reschedule {task_name} on {date} to {new_time}", ["task_name", "date", "new_time"]),
    ("set {task_name} on {date} to {new_time}", ["task_name", "date", "new_time"]),

    # ================================================================
    # SINGLE FIELD — new_duration only
    # ================================================================

    # task + new_duration
    ("change {task_name} duration to {new_duration}", ["task_name", "new_duration"]),
    ("update {task_name} duration to {new_duration}", ["task_name", "new_duration"]),
    ("set {task_name} duration to {new_duration}", ["task_name", "new_duration"]),
    ("update {task_name} to {new_duration}", ["task_name", "new_duration"]),
    ("change the duration of {task_name} to {new_duration}", ["task_name", "new_duration"]),
    ("update the duration of {task_name} to {new_duration}", ["task_name", "new_duration"]),
    ("{task_name} will take {new_duration} update it", ["task_name", "new_duration"]),
    ("I want to change {task_name} duration to {new_duration}", ["task_name", "new_duration"]),
    ("can you set {task_name} duration to {new_duration}", ["task_name", "new_duration"]),
    ("update duration to {new_duration} for {task_name}", ["task_name", "new_duration"]),
    ("change duration to {new_duration} for {task_name}", ["task_name", "new_duration"]),
    ("the duration of {task_name} should be {new_duration}", ["task_name", "new_duration"]),

    # date + new_duration
    ("change task on {date} duration to {new_duration}", ["date", "new_duration"]),
    ("update task on {date} to last {new_duration}", ["date", "new_duration"]),
    ("set task on {date} duration to {new_duration}", ["date", "new_duration"]),

    # task + date + new_duration
    ("change {task_name} on {date} duration to {new_duration}", ["task_name", "date", "new_duration"]),
    ("update {task_name} on {date} to last {new_duration}", ["task_name", "date", "new_duration"]),
    ("set {task_name} on {date} duration to {new_duration}", ["task_name", "date", "new_duration"]),

    # ================================================================
    # SINGLE FIELD — new_taskname only
    # ================================================================

    # task + new_taskname
    ("rename {task_name} to {new_taskname}", ["task_name", "new_taskname"]),
    ("change {task_name} name to {new_taskname}", ["task_name", "new_taskname"]),
    ("update {task_name} name to {new_taskname}", ["task_name", "new_taskname"]),
    ("rename {task_name} as {new_taskname}", ["task_name", "new_taskname"]),
    ("change the name of {task_name} to {new_taskname}", ["task_name", "new_taskname"]),
    ("update the name of {task_name} to {new_taskname}", ["task_name", "new_taskname"]),
    ("call {task_name} {new_taskname} instead", ["task_name", "new_taskname"]),
    ("I want to rename {task_name} to {new_taskname}", ["task_name", "new_taskname"]),
    ("can you rename {task_name} to {new_taskname}", ["task_name", "new_taskname"]),
    ("change task name from {task_name} to {new_taskname}", ["task_name", "new_taskname"]),

    # date + new_taskname
    ("rename task on {date} to {new_taskname}", ["date", "new_taskname"]),
    ("change task on {date} name to {new_taskname}", ["date", "new_taskname"]),
    ("update task on {date} name to {new_taskname}", ["date", "new_taskname"]),

    # task + date + new_taskname
    ("rename {task_name} on {date} to {new_taskname}", ["task_name", "date", "new_taskname"]),
    ("change {task_name} on {date} name to {new_taskname}", ["task_name", "date", "new_taskname"]),

    # ================================================================
    # MULTI FIELD — new_date + new_time
    # ================================================================

    # task + new_date + new_time
    ("reschedule {task_name} to {new_date} at {new_time}", ["task_name", "new_date", "new_time"]),
    ("move {task_name} to {new_date} at {new_time}", ["task_name", "new_date", "new_time"]),
    ("change {task_name} to {new_date} at {new_time}", ["task_name", "new_date", "new_time"]),
    ("update {task_name} to {new_date} at {new_time}", ["task_name", "new_date", "new_time"]),
    ("shift {task_name} to {new_date} at {new_time}", ["task_name", "new_date", "new_time"]),
    ("postpone {task_name} to {new_date} at {new_time}", ["task_name", "new_date", "new_time"]),
    ("set {task_name} for {new_date} at {new_time}", ["task_name", "new_date", "new_time"]),
    ("I want to move {task_name} to {new_date} at {new_time}", ["task_name", "new_date", "new_time"]),
    ("can you reschedule {task_name} to {new_date} at {new_time}", ["task_name", "new_date", "new_time"]),
    ("please move {task_name} to {new_date} at {new_time}", ["task_name", "new_date", "new_time"]),

    # task + old_date + new_date + new_time
    ("move {task_name} from {date} to {new_date} at {new_time}", ["task_name", "date", "new_date", "new_time"]),
    ("reschedule {task_name} from {date} to {new_date} at {new_time}", ["task_name", "date", "new_date", "new_time"]),
    ("change {task_name} on {date} to {new_date} at {new_time}", ["task_name", "date", "new_date", "new_time"]),

    # task + old_time + new_date + new_time
    ("move {task_name} at {time} to {new_date} at {new_time}", ["task_name", "time", "new_date", "new_time"]),
    ("reschedule {task_name} at {time} to {new_date} at {new_time}", ["task_name", "time", "new_date", "new_time"]),

    # date + new_date + new_time
    ("move task on {date} to {new_date} at {new_time}", ["date", "new_date", "new_time"]),
    ("reschedule task on {date} to {new_date} at {new_time}", ["date", "new_date", "new_time"]),
    ("change task on {date} to {new_date} at {new_time}", ["date", "new_date", "new_time"]),

    # ================================================================
    # MULTI FIELD — new_date + new_priority
    # ================================================================

    # task + new_date + new_priority
    ("move {task_name} to {new_date} and change priority to {new_priority}", ["task_name", "new_date", "new_priority"]),
    ("reschedule {task_name} to {new_date} and set it as {new_priority}", ["task_name", "new_date", "new_priority"]),
    ("update {task_name} to {new_date} with {new_priority} priority", ["task_name", "new_date", "new_priority"]),
    ("change {task_name} date to {new_date} and priority to {new_priority}", ["task_name", "new_date", "new_priority"]),
    ("move {task_name} to {new_date} with {new_priority} priority", ["task_name", "new_date", "new_priority"]),

    # task + old_date + new_date + new_priority
    ("move {task_name} from {date} to {new_date} and mark it {new_priority}", ["task_name", "date", "new_date", "new_priority"]),
    ("reschedule {task_name} from {date} to {new_date} with {new_priority} priority", ["task_name", "date", "new_date", "new_priority"]),

    # ================================================================
    # MULTI FIELD — new_time + new_priority
    # ================================================================

    # task + new_time + new_priority
    ("change {task_name} time to {new_time} and priority to {new_priority}", ["task_name", "new_time", "new_priority"]),
    ("update {task_name} to {new_time} with {new_priority} priority", ["task_name", "new_time", "new_priority"]),
    ("set {task_name} at {new_time} and mark it {new_priority}", ["task_name", "new_time", "new_priority"]),
    ("move {task_name} to {new_time} and change priority to {new_priority}", ["task_name", "new_time", "new_priority"]),

    # task + old_time + new_time + new_priority
    ("change {task_name} from {time} to {new_time} and priority to {new_priority}", ["task_name", "time", "new_time", "new_priority"]),
    ("move {task_name} from {time} to {new_time} with {new_priority} priority", ["task_name", "time", "new_time", "new_priority"]),

    # date + new_time + new_priority
    ("update task on {date} to {new_time} with {new_priority} priority", ["date", "new_time", "new_priority"]),
    ("change task on {date} time to {new_time} and priority to {new_priority}", ["date", "new_time", "new_priority"]),

    # ================================================================
    # MULTI FIELD — new_date + new_time + new_priority
    # ================================================================

    # task + new_date + new_time + new_priority
    ("reschedule {task_name} to {new_date} at {new_time} with {new_priority} priority", ["task_name", "new_date", "new_time", "new_priority"]),
    ("move {task_name} to {new_date} at {new_time} and mark it {new_priority}", ["task_name", "new_date", "new_time", "new_priority"]),
    ("update {task_name} to {new_date} at {new_time} with {new_priority} priority", ["task_name", "new_date", "new_time", "new_priority"]),
    ("change {task_name} to {new_date} at {new_time} priority {new_priority}", ["task_name", "new_date", "new_time", "new_priority"]),
    ("shift {task_name} to {new_date} at {new_time} and set it as {new_priority}", ["task_name", "new_date", "new_time", "new_priority"]),

    # task + old_date + new_date + new_time + new_priority
    ("move {task_name} from {date} to {new_date} at {new_time} with {new_priority} priority", ["task_name", "date", "new_date", "new_time", "new_priority"]),
    ("reschedule {task_name} from {date} to {new_date} at {new_time} and mark it {new_priority}", ["task_name", "date", "new_date", "new_time", "new_priority"]),

    # ================================================================
    # MULTI FIELD — new_status + new_priority
    # ================================================================

    # task + new_status + new_priority
    ("mark {task_name} as {new_status} and change priority to {new_priority}", ["task_name", "new_status", "new_priority"]),
    ("update {task_name} status to {new_status} and priority to {new_priority}", ["task_name", "new_status", "new_priority"]),
    ("set {task_name} as {new_status} with {new_priority} priority", ["task_name", "new_status", "new_priority"]),
    ("change {task_name} to {new_status} and {new_priority} priority", ["task_name", "new_status", "new_priority"]),

    # date + new_status + new_priority
    ("mark task on {date} as {new_status} and set it to {new_priority} priority", ["date", "new_status", "new_priority"]),
    ("update task on {date} status to {new_status} and priority to {new_priority}", ["date", "new_status", "new_priority"]),

    # ================================================================
    # MULTI FIELD — new_duration + new_date
    # ================================================================

    # task + new_date + new_duration
    ("move {task_name} to {new_date} and change duration to {new_duration}", ["task_name", "new_date", "new_duration"]),
    ("reschedule {task_name} to {new_date} for {new_duration}", ["task_name", "new_date", "new_duration"]),
    ("update {task_name} to {new_date} and set duration to {new_duration}", ["task_name", "new_date", "new_duration"]),

    # task + old_date + new_date + new_duration
    ("move {task_name} from {date} to {new_date} and change duration to {new_duration}", ["task_name", "date", "new_date", "new_duration"]),

    # ================================================================
    # MULTI FIELD — new_duration + new_time
    # ================================================================

    # task + new_time + new_duration
    ("change {task_name} time to {new_time} and duration to {new_duration}", ["task_name", "new_time", "new_duration"]),
    ("update {task_name} to {new_time} for {new_duration}", ["task_name", "new_time", "new_duration"]),
    ("set {task_name} at {new_time} for {new_duration}", ["task_name", "new_time", "new_duration"]),
    ("move {task_name} to {new_time} and update duration to {new_duration}", ["task_name", "new_time", "new_duration"]),

    # task + old_time + new_time + new_duration
    ("move {task_name} from {time} to {new_time} and set duration to {new_duration}", ["task_name", "time", "new_time", "new_duration"]),

    # ================================================================
    # MULTI FIELD — new_taskname + new_date
    # ================================================================

    # task + new_taskname + new_date
    ("rename {task_name} to {new_taskname} and move it to {new_date}", ["task_name", "new_taskname", "new_date"]),
    ("change {task_name} name to {new_taskname} and reschedule to {new_date}", ["task_name", "new_taskname", "new_date"]),
    ("update {task_name} name to {new_taskname} and date to {new_date}", ["task_name", "new_taskname", "new_date"]),

    # ================================================================
    # MULTI FIELD — new_taskname + new_priority
    # ================================================================

    # task + new_taskname + new_priority
    ("rename {task_name} to {new_taskname} and change priority to {new_priority}", ["task_name", "new_taskname", "new_priority"]),
    ("change {task_name} name to {new_taskname} and set priority to {new_priority}", ["task_name", "new_taskname", "new_priority"]),
    ("update {task_name} to {new_taskname} with {new_priority} priority", ["task_name", "new_taskname", "new_priority"]),

    # ================================================================
    # MULTI FIELD — new_date + new_time + new_duration
    # ================================================================

    # task + new_date + new_time + new_duration
    ("reschedule {task_name} to {new_date} at {new_time} for {new_duration}", ["task_name", "new_date", "new_time", "new_duration"]),
    ("move {task_name} to {new_date} at {new_time} and set duration to {new_duration}", ["task_name", "new_date", "new_time", "new_duration"]),
    ("update {task_name} to {new_date} at {new_time} lasting {new_duration}", ["task_name", "new_date", "new_time", "new_duration"]),
    ("change {task_name} to {new_date} at {new_time} for {new_duration}", ["task_name", "new_date", "new_time", "new_duration"]),

    # task + old_date + new_date + new_time + new_duration
    ("move {task_name} from {date} to {new_date} at {new_time} for {new_duration}", ["task_name", "date", "new_date", "new_time", "new_duration"]),

    # ================================================================
    # MULTI FIELD — new_date + new_time + new_priority + new_duration
    # ================================================================

    # task + new_date + new_time + new_priority + new_duration
    ("reschedule {task_name} to {new_date} at {new_time} with {new_priority} priority for {new_duration}", ["task_name", "new_date", "new_time", "new_priority", "new_duration"]),
    ("move {task_name} to {new_date} at {new_time} mark it {new_priority} and set duration to {new_duration}", ["task_name", "new_date", "new_time", "new_priority", "new_duration"]),
    ("update {task_name} to {new_date} at {new_time} {new_priority} priority lasting {new_duration}", ["task_name", "new_date", "new_time", "new_priority", "new_duration"]),

    # ================================================================
    # STATUS SHORTCUTS — natural expressions
    # ================================================================

    ("complete {task_name}", ["task_name", "new_status"]),
    ("finish {task_name}", ["task_name", "new_status"]),
    ("done with {task_name}", ["task_name", "new_status"]),
    ("I finished {task_name}", ["task_name", "new_status"]),
    ("I completed {task_name}", ["task_name", "new_status"]),
    ("I am done with {task_name}", ["task_name", "new_status"]),
    ("start {task_name}", ["task_name", "new_status"]),
    ("I started {task_name}", ["task_name", "new_status"]),
    ("begin {task_name}", ["task_name", "new_status"]),
    ("I am working on {task_name}", ["task_name", "new_status"]),
    ("mark {task_name} done", ["task_name", "new_status"]),
    ("mark {task_name} complete", ["task_name", "new_status"]),
    ("tick off {task_name}", ["task_name", "new_status"]),
    ("check off {task_name}", ["task_name", "new_status"]),
]


# ================================================================
# BEHAVIORAL INTENT
# (no entities — classification only)
# ================================================================

BEHAVIORAL_TEMPLATES = [

    # ── PRODUCTIVE HOURS — general ───────────────────────────────
    ("when am I most productive", []),
    ("what are my peak hours", []),
    ("when do I work best", []),
    ("what time am I most focused", []),
    ("show my productive hours", []),
    ("when am I most active", []),
    ("what are my best hours", []),
    ("show peak performance hours", []),
    ("when do I focus the best", []),
    ("what time do I usually complete tasks", []),
    ("show my productivity hours", []),
    ("when am I most efficient", []),
    ("what time of day am I most productive", []),
    ("show when I am most productive", []),
    ("what are my peak productivity hours", []),
    ("tell me my best working hours", []),
    ("when should I do important work", []),
    ("what time am I sharpest", []),

    # ── PRODUCTIVE HOURS — per weekday ───────────────────────────
    ("when am I most productive on {day}", ["day"]),
    ("what are my best hours on {day}", ["day"]),
    ("when do I work best on {day}", ["day"]),
    ("what time am I focused on {day}", ["day"]),
    ("show productive hours for {day}", ["day"]),
    ("when am I most efficient on {day}", ["day"]),
    ("what time should I work on {day}", ["day"]),
    ("when do I usually complete tasks on {day}", ["day"]),
    ("what are my peak hours on {day}", ["day"]),
    ("tell me my best hours on {day}", ["day"]),

    # ── PRODUCTIVE DAYS ──────────────────────────────────────────
    ("what are my most productive days", []),
    ("which days am I most focused", []),
    ("when do I work best during the week", []),
    ("show my best days", []),
    ("what days am I most efficient", []),
    ("which day of the week am I most productive", []),
    ("show productive days", []),
    ("what are my peak days", []),
    ("which days do I complete the most tasks", []),
    ("show my best performing days", []),
    ("what days should I schedule important tasks", []),
    ("which days am I most active", []),
    ("tell me my best days of the week", []),
    ("what day am I sharpest", []),

    ("find the best time on {day}", ["day"]),
    ("when should I schedule on {day}", ["day"]),
    ("what is the best time to study on {day}", ["day"]),
    ("suggest a time for {day}", ["day"]),
    ("what time works best on {day}", ["day"]),
    ("find me a good time on {day}", ["day"]),
    ("when should I work on {day}", ["day"]),
    ("what is the ideal time on {day}", ["day"]),
    ("suggest the best slot on {day}", ["day"]),
    ("when am I usually free on {day}", ["day"]),
    ("what time is best for me on {day}", ["day"]),
    ("when should I do my tasks on {day}", ["day"]),

    # # ── COMPLETION RATE / ON-TIME STATS ──────────────────────────
    ("show my completion rate", []),
    ("how many tasks do I complete on time", []),
    ("what is my on-time rate", []),
    ("show my task completion stats", []),
    ("how often do I finish tasks on time", []),
    ("what percentage of tasks do I complete", []),
    ("show how well I am doing", []),
    ("how many tasks did I complete this week", []),
    ("show my task history", []),
    ("what is my productivity score", []),
    ("how consistent am I with tasks", []),
    ("am I completing tasks on time", []),
    ("what is my task success rate", []),

    # # # ── FULL PRODUCTIVITY REPORT ─────────────────────────────────
    ("show my productivity report", []),
    ("show my productivity summary", []),
    ("give me a productivity overview", []),
    ("show my analytics", []),
    ("show my performance report", []),
    ("what does my productivity look like", []),
    ("give me a summary of my work habits", []),
    ("show my task analytics", []),
    ("how productive have I been", []),
    ("show my overall performance", []),
    ("give me a full report", []),
    ("show me how I am doing overall", []),
    ("what is my overall productivity", []),
    ("analyze my work patterns", []),
    ("show my study habits", []),
]
